In [ ]:
# DBS Optimization Benchmark (STN–GPe) — Step-by-step

This notebook is the **transparent** version of the project. It does:

1) Build an **oracle** over a strict DBS grid (Set 2) in parallel:  
   - `beta_ratio` = beta(DBS)/beta(baseline)  
   - `energy_proxy` = normalized A²·f·PW

2) Compute **Pareto front + knee** (beta vs energy).

3) Define a **scalar benchmark objective** using the knee energy budget:
   \[
   cost = beta\_ratio + penalty \cdot \max(0, energy - E_{knee})
   \]

4) Compute **ground truth** optimum for that scalar objective.

5) Run **budgeted optimizers** (Random, GA, Surrogate→QUBO exact; BO optional) and score each against ground truth using:
   - final best cost
   - final regret = best_found − cost_star


In [25]:
# =========================
# 0) CONFIG (edit here)
# =========================

import numpy as np
import time
from dataclasses import dataclass, asdict
from typing import Optional, Tuple, List, Dict

# Parallelism
MAX_WORKERS = 24
CHUNKSIZE = 32

# Simulation settings
@dataclass(frozen=True)
class SimParams:
    T: float = 2.0
    dt: float = 1e-4
    discard: float = 0.2
    seed: int = 0

SIM = SimParams(T=2.0, dt=1e-4, discard=0.2, seed=0)
BETA_BAND = (13.0, 30.0)

# Set 2 strict grid (BIGGER)
F_VALS = np.linspace(60, 180, 64)  # Hz (was 16)
A_VALS = np.linspace(0.0, 1.5, 64) # model units (was 16)

# PW: 16 bins from 50 to 240 us
PW_US  = np.linspace(50, 240, 16, dtype=float)
PW_VALS = PW_US * 1e-6  # seconds

Nf, Na, Npw = len(F_VALS), len(A_VALS), len(PW_VALS)
N_TOTAL = Nf * Na * Npw
print("Grid size:", (Nf, Na, Npw), "=", N_TOTAL)


# Benchmark settings
BUDGET = 40
SEEDS  = 10

# Scalar objective penalty for exceeding knee energy budget
PENALTY = 5.0

# Optional: Bayesian optimization
USE_BAYES_OPT = True


Grid size: (64, 64, 16) = 65536


In [9]:
# =========================
# 1) SIGNAL METRICS
# =========================

import numpy as np
from scipy.signal import welch

def beta_band_power(signal: np.ndarray, fs: float, band: Tuple[float, float] = (13.0, 30.0)) -> float:
    f, P = welch(signal, fs=fs, nperseg=min(len(signal), 8192))
    lo, hi = band
    mask = (f >= lo) & (f <= hi)
    return float(np.trapz(P[mask], f[mask]))

def estimate_peak_hz(signal: np.ndarray, fs: float) -> float:
    f, P = welch(signal, fs=fs, nperseg=min(len(signal), 8192))
    idx = np.argmax(P[1:]) + 1
    return float(f[idx])


In [10]:
# =========================
# 2) STN–GPe SIMULATOR
# =========================

import numpy as np
from dataclasses import dataclass
from typing import Optional, Tuple

def tanh_act(x: float, gain: float) -> float:
    return float(np.tanh(gain * x))

@dataclass(frozen=True)
class STNGPePatient:
    tau_stn: float
    tau_gpe: float
    delay_stn_to_gpe: float
    delay_gpe_to_stn: float
    w_stn_to_gpe: float
    w_gpe_to_stn: float
    I_cort: float
    I_gpe_bias: float
    gain_stn: float
    gain_gpe: float
    noise_std: float = 0.0

@dataclass(frozen=True)
class DBSParams:
    f_hz: float
    A: float
    pw_s: float

MAIN_PATIENT = STNGPePatient(
    tau_stn=0.009480108684355502,
    tau_gpe=0.01568342986890998,
    delay_stn_to_gpe=0.005856049933455449,
    delay_gpe_to_stn=0.004355096988711622,
    w_stn_to_gpe=0.8766085173129406,
    w_gpe_to_stn=0.8007248587605185,
    I_cort=0.21584549577209655,
    I_gpe_bias=0.0,
    gain_stn=2.418440027526686,
    gain_gpe=2.4195288944990088,
    noise_std=0.0
)

def simulate_stn_gpe(patient: STNGPePatient, dbs: Optional[DBSParams], sim: SimParams) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(sim.seed)
    n = int(sim.T / sim.dt)
    t = np.arange(n) * sim.dt

    stn = np.zeros(n, dtype=float)
    gpe = np.zeros(n, dtype=float)
    stn[0], gpe[0] = 0.1, 0.1

    d_sg = max(1, int(patient.delay_stn_to_gpe / sim.dt))
    d_gs = max(1, int(patient.delay_gpe_to_stn / sim.dt))

    if dbs is None or dbs.A == 0 or dbs.f_hz == 0 or dbs.pw_s == 0:
        f = 0.0; A = 0.0; pw = 0.0
    else:
        f = float(dbs.f_hz); A = float(dbs.A); pw = float(dbs.pw_s)

    period = 1.0 / f if f > 0 else None
    next_pulse = 0.0
    pulse_end = -1.0

    for k in range(n - 1):
        tt = t[k]

        Idbs = 0.0
        if f > 0:
            if tt >= next_pulse:
                pulse_end = next_pulse + pw
                next_pulse += period
            if tt < pulse_end:
                Idbs = A

        stn_del = stn[k - d_sg] if k >= d_sg else stn[0]
        gpe_del = gpe[k - d_gs] if k >= d_gs else gpe[0]

        ns = patient.noise_std * rng.standard_normal()
        ng = patient.noise_std * rng.standard_normal()

        inp_stn = patient.I_cort - patient.w_gpe_to_stn * gpe_del + Idbs + ns
        inp_gpe = patient.w_stn_to_gpe * stn_del + patient.I_gpe_bias + ng

        Fstn = tanh_act(inp_stn, patient.gain_stn)
        Fgpe = tanh_act(inp_gpe, patient.gain_gpe)

        dstn = (-stn[k] + Fstn) / patient.tau_stn
        dgpe = (-gpe[k] + Fgpe) / patient.tau_gpe

        stn[k + 1] = stn[k] + sim.dt * dstn
        gpe[k + 1] = gpe[k] + sim.dt * dgpe

    start = int(sim.discard / sim.dt)
    return t[start:], (stn - gpe)[start:]


In [26]:
# =========================
# 3) ORACLE (parallel)
# =========================

import multiprocessing as mp

def flat_index(fi:int, ai:int, pwi:int) -> int:
    return (fi * Na + ai) * Npw + pwi

def unflat_index(flat:int) -> Tuple[int,int,int]:
    pwi = flat % Npw
    flat //= Npw
    ai = flat % Na
    flat //= Na
    fi = flat
    return fi, ai, pwi

def baseline_beta(patient: STNGPePatient, sim: SimParams) -> Tuple[float, float]:
    t, y = simulate_stn_gpe(patient, dbs=None, sim=sim)
    fs = 1.0 / (t[1] - t[0])
    b0 = beta_band_power(y, fs, BETA_BAND)
    return float(fs), float(b0)

FS, B0 = baseline_beta(MAIN_PATIENT, SIM)
t_base, y_base = simulate_stn_gpe(MAIN_PATIENT, dbs=None, sim=SIM)
print("Baseline peak frequency:", estimate_peak_hz(y_base, FS))

F_MAX = float(F_VALS.max())
A_MAX = float(A_VALS.max()) if float(A_VALS.max()) > 0 else 1.0
PW_MAX = float(PW_VALS.max())

def energy_proxy(f_hz: float, A: float, pw_s: float) -> float:
    return float((A / A_MAX)**2 * (f_hz / F_MAX) * (pw_s / PW_MAX))

def worker_eval(task: Tuple[int,int,int]) -> Tuple[int,float,float]:
    fi, ai, pwi = task
    f = float(F_VALS[fi]); A = float(A_VALS[ai]); pw = float(PW_VALS[pwi])
    dbs = DBSParams(f_hz=f, A=A, pw_s=pw)
    t, y = simulate_stn_gpe(MAIN_PATIENT, dbs=dbs, sim=SIM)
    beta = beta_band_power(y, FS, BETA_BAND)
    beta_ratio = float(beta / (B0 + 1e-12))
    e = energy_proxy(f, A, pw)
    return flat_index(fi, ai, pwi), beta_ratio, e

def build_oracle_parallel():
    tasks = [(fi, ai, pwi) for fi in range(Nf) for ai in range(Na) for pwi in range(Npw)]
    workers = min(MAX_WORKERS, mp.cpu_count())
    print(f"Building oracle for {len(tasks)} points with {workers} workers...")

    try:
        ctx = mp.get_context("fork")
    except ValueError:
        ctx = mp.get_context("spawn")

    beta_ratio = np.zeros(N_TOTAL, dtype=float)
    energy = np.zeros(N_TOTAL, dtype=float)

    t0 = time.time()
    with ctx.Pool(processes=workers) as pool:
        for flat, br, e in pool.imap_unordered(worker_eval, tasks, chunksize=CHUNKSIZE):
            beta_ratio[flat] = br
            energy[flat] = e
    t1 = time.time()
    print(f"Oracle built in {t1-t0:.2f} seconds")
    return beta_ratio, energy

beta_ratio_flat, energy_flat = build_oracle_parallel()
print("beta_ratio range:", float(beta_ratio_flat.min()), "to", float(beta_ratio_flat.max()))
print("energy range:", float(energy_flat.min()), "to", float(energy_flat.max()))


Baseline peak frequency: 19.53125000000215
Building oracle for 65536 points with 24 workers...
Oracle built in 268.69 seconds
beta_ratio range: 0.746917735138448 to 1.0001569216698758
energy range: 0.0 to 1.0


In [27]:
# =========================
# 4) PARETO + KNEE
# =========================

def pareto_front_fast(beta: np.ndarray, energy: np.ndarray) -> np.ndarray:
    order = np.argsort(beta)
    keep = []
    best_e = float("inf")
    for idx in order:
        e = float(energy[idx])
        if e < best_e:
            keep.append(int(idx))
            best_e = e
    return np.array(keep, dtype=int)

def choose_knee(beta_front: np.ndarray, energy_front: np.ndarray) -> int:
    b_min, b_max = float(beta_front.min()), float(beta_front.max())
    e_min, e_max = float(energy_front.min()), float(energy_front.max())
    bn = (beta_front - b_min) / (b_max - b_min + 1e-12)
    en = (energy_front - e_min) / (e_max - e_min + 1e-12)
    dist = np.sqrt(bn**2 + en**2)
    return int(np.argmin(dist))

pareto_idx = pareto_front_fast(beta_ratio_flat, energy_flat)
beta_p = beta_ratio_flat[pareto_idx]
energy_p = energy_flat[pareto_idx]
knee_local = choose_knee(beta_p, energy_p)
knee_flat = int(pareto_idx[knee_local])
kfi, kai, kpwi = unflat_index(knee_flat)

knee = {
    "f_hz": float(F_VALS[kfi]),
    "A": float(A_VALS[kai]),
    "pw_us": float(PW_US[kpwi]),
    "beta_ratio": float(beta_ratio_flat[knee_flat]),
    "energy": float(energy_flat[knee_flat]),
}
print("Pareto points:", len(pareto_idx))
print("Knee point:", knee)


Pareto points: 109
Knee point: {'f_hz': 176.1904761904762, 'A': 0.6666666666666666, 'pw_us': 227.33333333333331, 'beta_ratio': 0.7844227120082152, 'energy': 0.1831457167533985}


In [14]:
#This is the ground truth!!!

In [29]:
# =========================
# 5) SCALAR OBJECTIVE + GROUND TRUTH
# =========================

def scalar_cost(beta_ratio: np.ndarray, energy: np.ndarray, E_max: float, penalty: float) -> np.ndarray:
    over = np.maximum(0.0, energy - E_max)
    return beta_ratio + penalty * over

E_KNEE = float(knee["energy"])
cost_flat = scalar_cost(beta_ratio_flat, energy_flat, E_max=E_KNEE, penalty=PENALTY)

best_flat = int(np.argmin(cost_flat))
bfi, bai, bpwi = unflat_index(best_flat)

ground_truth = {
    "E_knee": E_KNEE,
    "penalty": PENALTY,
    "cost_star": float(cost_flat[best_flat]),
    "best_setting": {
        "f_hz": float(F_VALS[bfi]),
        "A": float(A_VALS[bai]),
        "pw_us": float(PW_US[bpwi]),
        "beta_ratio": float(beta_ratio_flat[best_flat]),
        "energy": float(energy_flat[best_flat]),
    }
}

print("GROUND TRUTH (scalar objective):")
print(ground_truth)


GROUND TRUTH (scalar objective):
{'E_knee': 0.1831457167533985, 'penalty': 5.0, 'cost_star': 0.7844227120082152, 'best_setting': {'f_hz': 176.1904761904762, 'A': 0.6666666666666666, 'pw_us': 227.33333333333331, 'beta_ratio': 0.7844227120082152, 'energy': 0.1831457167533985}}


In [45]:
# =========================
# 6) OPTIMIZERS + scoring
# =========================

def score_evals(evals: List[Tuple[int,int,int,float]], cost_star: float) -> Dict[str, float]:
    best = float("inf")
    for _,_,_,c in evals:
        if c < best:
            best = c
    return {"final_best_cost": best, "final_regret": best - cost_star}

def run_random(cost_flat: np.ndarray, budget:int, seed:int) -> List[Tuple[int,int,int,float]]:
    rng = np.random.default_rng(seed)
    chosen = rng.choice(np.arange(N_TOTAL), size=min(budget, N_TOTAL), replace=False)
    return [( *unflat_index(int(flat)), float(cost_flat[int(flat)]) ) for flat in chosen]

def run_ga(cost_flat: np.ndarray, budget:int, seed:int, pop_size:int=40) -> List[Tuple[int,int,int,float]]:
    rng = np.random.default_rng(seed)
    bitlen = int(np.ceil(np.log2(N_TOTAL)))

    cache: Dict[int,float] = {}
    order: List[int] = []

    def cost_of(x:int) -> float:
        x = x % N_TOTAL
        if x not in cache:
            cache[x] = float(cost_flat[x])
            order.append(x)
        return cache[x]

    pop = [int(rng.integers(0, N_TOTAL)) for _ in range(pop_size)]
    for x in pop:
        if len(cache) >= budget: break
        cost_of(x)

    def tournament(k=3) -> int:
        cand = rng.choice(pop, size=k, replace=False)
        return int(min(cand, key=cost_of))

    def crossover(a:int, b:int) -> Tuple[int,int]:
        if rng.random() > 0.7:
            return a, b
        point = int(rng.integers(1, bitlen))
        mask = (1 << point) - 1
        a2 = (a & ~mask) | (b & mask)
        b2 = (b & ~mask) | (a & mask)
        return a2 % N_TOTAL, b2 % N_TOTAL

    def mutate(x:int, p:float) -> int:
        for bit in range(bitlen):
            if rng.random() < p:
                x ^= (1 << bit)
        return x % N_TOTAL

    p_mut = 1.0 / max(bitlen, 1)

    while len(cache) < budget:
        new_pop = []
        elite = sorted(pop, key=cost_of)[:2]
        new_pop.extend(elite)
        while len(new_pop) < pop_size:
            p1 = tournament()
            p2 = tournament()
            c1, c2 = crossover(p1, p2)
            c1 = mutate(c1, p_mut)
            c2 = mutate(c2, p_mut)
            new_pop.extend([c1, c2])
        pop = new_pop[:pop_size]
        for x in pop:
            if len(cache) >= budget: break
            cost_of(x)

    evals = []
    for flat in order:
        fi, ai, pwi = unflat_index(flat)
        evals.append((fi, ai, pwi, float(cost_flat[flat])))
    return evals

def run_surrogate_exact(cost_flat: np.ndarray, budget:int, seed:int) -> List[Tuple[int,int,int,float]]:
    rng = np.random.default_rng(seed)
    bitlen = int(np.ceil(np.log2(N_TOTAL)))

    def flat_to_bits(flat:int) -> np.ndarray:
        return np.array([(flat >> (bitlen-1-i)) & 1 for i in range(bitlen)], dtype=float)

    def quad_design(B: np.ndarray) -> np.ndarray:
        m, n = B.shape
        cols = 1 + n + (n*(n-1))//2
        Phi = np.zeros((m, cols))
        Phi[:,0] = 1.0
        Phi[:,1:1+n] = B
        col = 1+n
        for i in range(n):
            for j in range(i+1, n):
                Phi[:,col] = B[:,i]*B[:,j]
                col += 1
        return Phi

    evaluated = set()
    evals: List[Tuple[int,int,int,float]] = []

    n_init = max(10, budget//4)
    n_init = min(n_init, budget-1)
    init = rng.choice(np.arange(N_TOTAL), size=n_init, replace=False)
    for flat in init:
        evaluated.add(int(flat))
        fi, ai, pwi = unflat_index(int(flat))
        evals.append((fi, ai, pwi, float(cost_flat[int(flat)])))

    while len(evals) < budget:
        flats = [flat_index(fi, ai, pwi) for fi, ai, pwi, _ in evals]
        Bmat = np.vstack([flat_to_bits(int(f)) for f in flats])
        y = np.array([c for *_, c in evals], dtype=float)

        Phi = quad_design(Bmat)
        coef, *_ = np.linalg.lstsq(Phi, y, rcond=None)

        best_pred = float("inf")
        best_flat = None
        for flat in range(N_TOTAL):
            if flat in evaluated:
                continue
            b = flat_to_bits(flat)
            pred = coef[0] + np.dot(coef[1:1+bitlen], b)
            col = 1+bitlen
            for i in range(bitlen):
                for j in range(i+1, bitlen):
                    pred += coef[col] * b[i]*b[j]
                    col += 1
            if pred < best_pred:
                best_pred = float(pred)
                best_flat = flat

        if best_flat is None:
            break

        evaluated.add(best_flat)
        fi, ai, pwi = unflat_index(best_flat)
        evals.append((fi, ai, pwi, float(cost_flat[int(best_flat)])))

    return evals

def run_method_summary(method_fn, name:str):
    finals = []
    regrets = []
    for s in range(SEEDS):
        evals = method_fn(cost_flat, BUDGET, s)
        scored = score_evals(evals, ground_truth["cost_star"])
        finals.append(scored["final_best_cost"])
        regrets.append(scored["final_regret"])
    print(f"{name:20s} mean_final_cost={float(np.mean(finals)):.4f}  mean_regret={float(np.mean(regrets)):.4f}  std_cost={float(np.std(finals)):.4f}")

run_method_summary(run_random, "random")
run_method_summary(run_ga, "ga")
run_method_summary(run_surrogate_exact, "surrogate_exact")


random               mean_final_cost=0.8503  mean_regret=0.0659  std_cost=0.0239
ga                   mean_final_cost=0.8392  mean_regret=0.0548  std_cost=0.0249
surrogate_exact      mean_final_cost=0.8418  mean_regret=0.0574  std_cost=0.0451


In [19]:
import sklearn

In [33]:
# =========================
# Bayesian Optimization (deterministic GP; no WhiteKernel spam)
# =========================

import numpy as np

def run_bo(cost_flat: np.ndarray, budget:int, seed:int, n_init:int=12, xi:float=0.01):
    """
    BO on a DISCRETE grid by modeling (fi,ai,pwi) as continuous in [0,1]^3 and
    selecting the next point via Expected Improvement (EI).

    Deterministic oracle fix:
      - No WhiteKernel
      - alpha ~ 1e-10
    """
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import Matern, ConstantKernel
    from scipy.stats import norm
    import warnings
    from sklearn.exceptions import ConvergenceWarning

    warnings.filterwarnings("ignore", category=ConvergenceWarning)

    rng = np.random.default_rng(seed)

    # Build all candidate points in normalized coordinates
    X_all = []
    y_all = []
    idx_all = []
    for fi in range(Nf):
        for ai in range(Na):
            for pwi in range(Npw):
                X_all.append([fi/(Nf-1), ai/(Na-1), pwi/(Npw-1)])
                idx_all.append((fi, ai, pwi))
                y_all.append(float(cost_flat[flat_index(fi,ai,pwi)]))

    X_all = np.array(X_all, dtype=float)
    y_all = np.array(y_all, dtype=float)

    remaining = np.arange(len(idx_all))
    rng.shuffle(remaining)

    # init observations
    obs_idx = list(remaining[:n_init])
    remaining = remaining[n_init:]

    X_obs = X_all[obs_idx].copy()
    y_obs = y_all[obs_idx].copy()
    evals = [(idx_all[i][0], idx_all[i][1], idx_all[i][2], float(y_all[i])) for i in obs_idx]

    def EI(mu, sigma, best):
        sigma = np.maximum(sigma, 1e-12)
        imp = best - mu - xi  # minimization
        Z = imp / sigma
        return imp * norm.cdf(Z) + sigma * norm.pdf(Z)

    while len(evals) < budget and len(remaining) > 0:
        kernel = ConstantKernel(1.0, constant_value_bounds=(1e-3, 1e3)) * Matern(nu=2.5)
        gp = GaussianProcessRegressor(
            kernel=kernel,
            alpha=1e-10,          # deterministic jitter
            normalize_y=True,
            random_state=seed,
            n_restarts_optimizer=2
        )
        gp.fit(X_obs, y_obs)

        X_rem = X_all[remaining]
        mu, std = gp.predict(X_rem, return_std=True)
        best = float(np.min(y_obs))
        ei = EI(mu, std, best)

        pick_local = int(np.argmax(ei))
        pick = int(remaining[pick_local])
        remaining = np.delete(remaining, pick_local)

        X_obs = np.vstack([X_obs, X_all[pick]])
        y_obs = np.append(y_obs, y_all[pick])

        fi, ai, pwi = idx_all[pick]
        evals.append((fi, ai, pwi, float(y_all[pick])))

    return evals


def run_method_summary(method_fn, name:str):
    finals, regrets = [], []
    for s in range(SEEDS):
        evals = method_fn(cost_flat, BUDGET, s)
        best = min([c for *_, c in evals])
        finals.append(best)
        regrets.append(best - ground_truth["cost_star"])
    print(f"{name:20s} mean_final_cost={float(np.mean(finals)):.4f}  mean_regret={float(np.mean(regrets)):.4f}  std_cost={float(np.std(finals)):.4f}")

# Example:
run_method_summary(run_bo, "bo")


bo                   mean_final_cost=0.8386  mean_regret=0.0541  std_cost=0.0356


In [34]:
#For CMA-ES

In [38]:
# =========================
# CMA-ES (continuous -> rounded to grid)
# =========================

def run_cma_es(cost_flat: np.ndarray, budget:int, seed:int, sigma0:float=0.25, popsize:int=12):
    """
    CMA-ES searches in continuous x in [0,1]^3, then we round to (fi,ai,pwi).
    Each UNIQUE rounded point counts as one evaluation.
    """
    try:
        import cma
    except Exception as e:
        raise RuntimeError("Install CMA-ES library first: pip install cma") from e

    rng = np.random.default_rng(seed)

    # cache unique evaluations so we don't waste budget on duplicates
    cache = {}
    evals = []

    def x_to_idx(x):
        # clip to [0,1], then map to grid indices
        x = np.clip(x, 0.0, 1.0)
        fi = int(round(x[0] * (Nf-1)))
        ai = int(round(x[1] * (Na-1)))
        pwi = int(round(x[2] * (Npw-1)))
        return fi, ai, pwi

    def evaluate_x(x):
        fi, ai, pwi = x_to_idx(x)
        flat = flat_index(fi, ai, pwi)
        if flat not in cache and len(cache) < budget:
            cache[flat] = float(cost_flat[flat])
            evals.append((fi, ai, pwi, float(cost_flat[flat])))
        # return cost even if already evaluated (CMA needs a value)
        return float(cost_flat[flat])

    # CMA-ES setup
    x0 = [0.5, 0.5, 0.5]  # start center
    es = cma.CMAEvolutionStrategy(
        x0, sigma0,
        {"seed": seed, "popsize": popsize, "bounds": [0.0, 1.0], "verbose": -9}
    )

    while not es.stop() and len(cache) < budget:
        X = es.ask()
        y = [evaluate_x(x) for x in X]
        es.tell(X, y)

    return evals

# Example:
run_method_summary(run_cma_es, "cma_es")


cma_es               mean_final_cost=0.8317  mean_regret=0.0473  std_cost=0.0138


In [39]:
#Quantum ver

In [48]:
# =========================
# Surrogate -> QUBO -> QAOA
# =========================

def run_surrogate_qaoa(cost_flat: np.ndarray, budget:int, seed:int, p_layers:int=1, shots:int=1024, maxiter:int=60, ridge:float=1e-3, max_pairs:int=300):
    """
    Hybrid quantum pipeline:
      1) sample some points (evaluations)
      2) fit quadratic surrogate in bitspace (ridge regression)
      3) convert surrogate to sparse QUBO (keep top |b_ij| pairs)
      4) use QAOA to sample good bitstrings
      5) evaluate best unseen bitstring on true oracle
      repeat until budget

    Requires qiskit + qiskit-aer.
    """
    try:
        import numpy as np
        from scipy.optimize import minimize
        from qiskit import QuantumCircuit
        from qiskit_aer.primitives import Sampler
    except Exception as e:
        raise RuntimeError("Install: pip install qiskit qiskit-aer") from e

    rng = np.random.default_rng(seed)
    N = cost_flat.size
    bitlen = int(np.ceil(np.log2(N)))

    # helper: encode flat -> bits (MSB->LSB)
    def flat_to_bits(flat:int) -> np.ndarray:
        return np.array([(flat >> (bitlen-1-i)) & 1 for i in range(bitlen)], dtype=float)

    # quadratic design matrix for ridge regression
    def quad_design(B: np.ndarray) -> np.ndarray:
        m, n = B.shape
        cols = 1 + n + (n*(n-1))//2
        Phi = np.zeros((m, cols))
        Phi[:,0] = 1.0
        Phi[:,1:1+n] = B
        col = 1+n
        for i in range(n):
            for j in range(i+1, n):
                Phi[:,col] = B[:,i]*B[:,j]
                col += 1
        return Phi

    # fit ridge regression: (Phi^T Phi + ridge I) theta = Phi^T y
    def fit_surrogate(evals):
        flats = [flat_index(fi, ai, pwi) for fi, ai, pwi, _ in evals]
        Bmat = np.vstack([flat_to_bits(int(f)) for f in flats])
        y = np.array([c for *_, c in evals], dtype=float)

        Phi = quad_design(Bmat)
        I = np.eye(Phi.shape[1])
        theta = np.linalg.solve(Phi.T @ Phi + ridge * I, Phi.T @ y)

        b0 = float(theta[0])
        b_lin = theta[1:1+bitlen].copy()
        b_pair = np.zeros((bitlen, bitlen), dtype=float)
        col = 1+bitlen
        for i in range(bitlen):
            for j in range(i+1, bitlen):
                b_pair[i,j] = float(theta[col])
                col += 1
        return b0, b_lin, b_pair

    # keep only strongest pair couplings to make circuit smaller/faster
    def sparsify_pairs(b_pair: np.ndarray) -> List[Tuple[int,int,float]]:
        pairs = []
        for i in range(bitlen):
            for j in range(i+1, bitlen):
                w = b_pair[i,j]
                if w != 0.0:
                    pairs.append((i,j,w))
        pairs.sort(key=lambda x: abs(x[2]), reverse=True)
        return pairs[:max_pairs]

    sampler = Sampler()

    # QAOA circuit: H -> cost phases -> mixer phases -> measure
    def build_qaoa_circuit(gammas, betas, b_lin, sparse_pairs):
        qc = QuantumCircuit(bitlen)
        qc.h(range(bitlen))

        for layer in range(p_layers):
            g = gammas[layer]
            b = betas[layer]
            # linear Z phases
            for i in range(bitlen):
                if b_lin[i] != 0:
                    qc.rz(2 * g * b_lin[i], i)
            # pair ZZ phases
            for i, j, w in sparse_pairs:
                qc.rzz(2 * g * w, i, j)
            # mixer
            for i in range(bitlen):
                qc.rx(2 * b, i)

        qc.measure_all()
        return qc

    def surrogate_value(bits: np.ndarray, b0, b_lin, b_pair):
        val = b0 + float(np.dot(b_lin, bits))
        for i in range(bitlen):
            for j in range(i+1, bitlen):
                val += float(b_pair[i,j] * bits[i]*bits[j])
        return float(val)

    # Track evaluated points
    evaluated = set()
    evals: List[Tuple[int,int,int,float]] = []

    # init samples
    n_init = max(10, budget//4)  # a bit higher than 30 for stability
    n_init = n_init = min(n_init, budget-1)
    init = rng.choice(np.arange(N), size=n_init, replace=False)
    for flat in init:
        evaluated.add(int(flat))
        fi, ai, pwi = unflat_index(int(flat))
        evals.append((fi, ai, pwi, float(cost_flat[int(flat)])))

    while len(evals) < budget:
        b0, b_lin, b_pair = fit_surrogate(evals)
        sparse_pairs = sparsify_pairs(b_pair)

        # objective: expected surrogate cost, with huge penalty if already evaluated
        def expected_cost(params):
            gammas = params[:p_layers]
            betas = params[p_layers:]
            qc = build_qaoa_circuit(gammas, betas, b_lin, sparse_pairs)
            res = sampler.run([qc], shots=shots).result()
            dist = res.quasi_dists[0]

            exp = 0.0
            for bit_int, prob in dist.items():
                flat = int(bit_int)
                if flat >= N:
                    # ignore out-of-range bitstrings if N not power of 2
                    continue
                if flat in evaluated:
                    exp += prob * 1e3
                else:
                    bits = flat_to_bits(flat)
                    exp += prob * surrogate_value(bits, b0, b_lin, b_pair)
            return float(exp)

        x0 = np.concatenate([
            rng.uniform(0, np.pi, size=p_layers),
            rng.uniform(0, np.pi, size=p_layers),
        ])

        res = minimize(expected_cost, x0=x0, method="COBYLA", options={"maxiter": maxiter})

        gammas = res.x[:p_layers]
        betas = res.x[p_layers:]
        qc = build_qaoa_circuit(gammas, betas, b_lin, sparse_pairs)

        # sample final circuit more to pick best new candidate
        res2 = sampler.run([qc], shots=4*shots).result()
        dist2 = res2.quasi_dists[0]

        best_flat = None
        best_pred = float("inf")
        for bit_int, prob in dist2.items():
            flat = int(bit_int)
            if flat >= N or flat in evaluated:
                continue
            bits = flat_to_bits(flat)
            pred = surrogate_value(bits, b0, b_lin, b_pair)
            if pred < best_pred:
                best_pred = float(pred)
                best_flat = flat

        # fallback: random unseen if QAOA doesn't propose anything new
        if best_flat is None:
            candidates = list(set(range(N)) - evaluated)
            if not candidates:
                break
            best_flat = int(rng.choice(candidates))

        evaluated.add(best_flat)
        fi, ai, pwi = unflat_index(best_flat)
        evals.append((fi, ai, pwi, float(cost_flat[best_flat])))

    return evals

# Example:
#run_method_summary(lambda cf,b,s: run_surrogate_qaoa(cf,b,s,p_layers=1), "surrogate_qaoa_p1")
#run_method_summary(lambda cf,b,s: run_surrogate_qaoa(cf,b,s,p_layers=2), "surrogate_qaoa_p2")
run_method_summary(lambda cf,b,s: run_surrogate_qaoa(cf,b,s,p_layers=1, shots=128, maxiter=15, max_pairs=50), "surrogate_qaoa_p1_fast")


/tmp/ipykernel_1400/1310132293.py:190: DeprecationWarning: Sampler has been deprecated as of Aer 0.15, please use SamplerV2 instead.
  run_method_summary(lambda cf,b,s: run_surrogate_qaoa(cf,b,s,p_layers=1, shots=128, maxiter=15, max_pairs=50), "surrogate_qaoa_p1_fast")


surrogate_qaoa_p1_fast mean_final_cost=0.8489  mean_regret=0.0645  std_cost=0.0252


In [42]:
import qiskit